# 02b — Method 3: SBERT Direction Embedding (Zero-LLM Baseline)

Fully non-LLM pipeline using sentence-transformers for both feature selection and per-feature direction inference.

**Pipeline:**
1. **Feature Selection (SBERT):** Embed prompt + feature descriptions, pick top-N by cosine similarity
2. **Direction Detection (SBERT):** For each selected feature, compare `sim(prompt, "high {feature}")` vs `sim(prompt, "low {feature}")` to determine per-feature direction
3. **Ranking (hardcoded):** Percentile-rank songs on selected features with inferred directions, weighted by relevance

**Prerequisites:** Run `02a_data_feature_prep.ipynb` first.

In [10]:
# --- Shared Setup ---
# This notebook depends on the data and features prepared in 02a.
# Run 02a_data_feature_prep.ipynb first, or load saved artifacts.

import sys
sys.path.insert(0, '..')

import os
import json
import time
import warnings
from pathlib import Path
from typing import TypedDict, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import cosine as cosine_dist
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore', category=FutureWarning)

from atdj.config import PROCESSED_DIR, DATA_DIR

# --- Load saved artifacts from notebook 02a ---
FEATURES_DIR = Path(PROCESSED_DIR) / 'features_exp'
df_merged = pd.read_pickle(FEATURES_DIR / 'df_merged.pkl')
print(f'Loaded df_merged: {df_merged.shape}')

import pickle
with open(FEATURES_DIR / 'feature_catalog.pkl', 'rb') as f:
    FEATURE_CATALOG = pickle.load(f)

ALL_FEATURE_NAMES = []
FEATURE_DESC_MAP = {}
for group, features in FEATURE_CATALOG.items():
    for fname, fdesc in features:
        ALL_FEATURE_NAMES.append(fname)
        FEATURE_DESC_MAP[fname] = fdesc
AVAILABLE_FEATURES = [f for f in ALL_FEATURE_NAMES if f in df_merged.columns]

with open(FEATURES_DIR / 'feature_prompt.txt', encoding='utf-8') as f:
    FEATURE_PROMPT = f.read()

print(f'Feature catalog: {len(AVAILABLE_FEATURES)} available features across {len(FEATURE_CATALOG)} groups')

Loaded df_merged: (184, 213)
Feature catalog: 99 available features across 15 groups


In [11]:
# --- Sentence-Transformers Setup ---
try:
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
    print('sentence-transformers loaded: all-MiniLM-L6-v2')
except ImportError:
    SBERT_AVAILABLE = False
    sbert_model = None
    print('sentence-transformers not available — cannot run Method 3')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sentence-transformers loaded: all-MiniLM-L6-v2


In [12]:
# Standardized output format — all methods return this structure
class SongMatch(TypedDict):
    filename: str
    score: float              # 0-1, higher = better match
    matched_features: dict    # feature_name: actual_value
    explanation: str          # why this song matches

class MatchResult(TypedDict):
    prompt: str
    method: str               # 'clap' | 'cot_llm_a' | 'cot_llm_b' | 'small_model' | 'sbert_direction'
    top_k_songs: list         # list of SongMatch
    bottom_k_songs: list      # worst-matching songs for human validation
    feature_ranges_used: dict # feature_name: {min, max, direction} (methods 2 & 3)
    metadata: dict            # method-specific (model name, latency, etc.)


# Test prompts used across all methods
TEST_PROMPTS = [
    'energetic tango with strong rhythm for experienced dancers',
    'melancholic and slow, perfect for a late-night vals',
    'bright and cheerful milonga with clear melody',
    'dramatic tango with heavy bandoneon and dark mood',
    'smooth and relaxed, good for warming up the floor',
    'classic golden-age tango from the 40s, warm and nostalgic',
    'a lively vals from the 50s with a strong orchestra',
    'need a cortina — something upbeat and non-tango to reset the floor',
]

TOP_K = 10
BOTTOM_K = 5

# Collect results
ALL_RESULTS: dict[str, list[MatchResult]] = {}

print(f'Test prompts: {len(TEST_PROMPTS)}')
print(f'Top-K: {TOP_K}')
for i, p in enumerate(TEST_PROMPTS, 1):
    print(f'  {i}. "{p}"')

Test prompts: 8
Top-K: 10
  1. "energetic tango with strong rhythm for experienced dancers"
  2. "melancholic and slow, perfect for a late-night vals"
  3. "bright and cheerful milonga with clear melody"
  4. "dramatic tango with heavy bandoneon and dark mood"
  5. "smooth and relaxed, good for warming up the floor"
  6. "classic golden-age tango from the 40s, warm and nostalgic"
  7. "a lively vals from the 50s with a strong orchestra"
  8. "need a cortina — something upbeat and non-tango to reset the floor"


---
## Step 1: Feature Selection (SBERT)

Embed the user prompt and all feature descriptions with sentence-transformers, then select the top-N features by cosine similarity. This step determines *which* features are relevant to the prompt — no direction inference yet.

In [13]:
# --- Step 1: SBERT Feature Selection ---

def embed_feature_descriptions(feature_catalog: dict, embedder) -> dict[str, np.ndarray]:
    """Embed each feature name+description with sentence-transformers.
    Returns {feature_name: embedding}."""
    names_and_descs = []
    feature_names = []
    for group, features in feature_catalog.items():
        for fname, fdesc in features:
            if fname in AVAILABLE_FEATURES:
                names_and_descs.append(f'{group}: {fname} — {fdesc}')
                feature_names.append(fname)

    embeddings = embedder.encode(names_and_descs, show_progress_bar=False)
    return {name: emb for name, emb in zip(feature_names, embeddings)}


def sbert_feature_select(prompt: str, feature_embeddings: dict,
                         embedder, top_n: int = 8) -> list[tuple[str, float]]:
    """Select top-N features by semantic similarity to the prompt.
    Returns list of (feature_name, similarity_score) sorted descending."""
    prompt_emb = embedder.encode([prompt])[0]

    similarities = {}
    for fname, femb in feature_embeddings.items():
        sim = 1.0 - cosine_dist(prompt_emb, femb)
        similarities[fname] = sim

    sorted_feats = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    return sorted_feats[:top_n]


# Pre-compute feature embeddings
if SBERT_AVAILABLE:
    feature_embeddings = embed_feature_descriptions(FEATURE_CATALOG, sbert_model)
    print(f'Embedded {len(feature_embeddings)} feature descriptions')

    # Test on first prompt
    test_feats = sbert_feature_select(TEST_PROMPTS[0], feature_embeddings, sbert_model)
    print(f'\nTest: "{TEST_PROMPTS[0]}"')
    print('Selected features (by relevance):')
    for fname, score in test_feats:
        print(f'  {score:.4f}  {fname} — {FEATURE_DESC_MAP.get(fname, "")}')

Embedded 99 feature descriptions

Test: "energetic tango with strong rhythm for experienced dancers"
Selected features (by relevance):
  0.6600  style — Tango dance style: tango, vals, milonga, or cortina
  0.5245  danceability — Algorithmic danceability score (0-1)
  0.4814  is_danceable — Binary danceability classification confidence
  0.4446  duration — Track duration in seconds (Essentia)
  0.4283  jamendo_dance — Dance tag activation
  0.4024  bpm — Beats per minute — tempo of the track (Essentia)
  0.3960  tempogram_global_mean — Mean tempogram value — tempo stability
  0.3706  onset_rate — Number of note onsets per second — rhythmic density


---
## Step 2: Per-Feature Direction Detection (SBERT)

For each selected feature, infer whether the prompt wants a **higher** or **lower** value.

**Approach:** Use two fixed **anchor sentences** — one representing high-energy/positive musical qualities and one representing low-energy/calm qualities:
- `HIGH_ANCHOR`: "energetic intense powerful strong fast loud bright happy upbeat"
- `LOW_ANCHOR`: "slow gentle calm soft quiet melancholic sad relaxed mellow"

1. Determine the **prompt pole**: is the prompt closer to the HIGH or LOW anchor?
2. Determine each **feature's pole**: is the feature description closer to the HIGH or LOW anchor?
3. If the prompt and feature are on the **same pole** → higher feature value = better match (direction = +1)
4. If they are on **opposite poles** → lower feature value = better match (direction = −1)

**Known limitation:** This binary classification conflates energy and valence — a "dramatic dark mood" registers as HIGH-pole, which may misclassify features like `mood_sad`. This is an inherent trade-off of a zero-LLM approach, and comparing against Method 2A (where the LLM reasons per-feature) quantifies the cost.

In [14]:
# --- Step 2: Per-Feature Direction via Anchor Pole Comparison ---

# Fixed anchor sentences representing two poles of musical qualities
HIGH_ANCHOR = 'energetic intense powerful strong fast loud bright happy upbeat'
LOW_ANCHOR = 'slow gentle calm soft quiet melancholic sad relaxed mellow'


def compute_feature_poles(feature_catalog: dict, available_features: list,
                          feature_desc_map: dict, embedder) -> dict[str, dict]:
    """Pre-compute the pole (HIGH or LOW) for each feature by comparing
    its description embedding against the two anchor embeddings.

    Returns {feature_name: {'pole': 'HIGH'|'LOW', 'sim_high': float, 'sim_low': float}}
    """
    high_anchor_emb = embedder.encode([HIGH_ANCHOR])[0]
    low_anchor_emb = embedder.encode([LOW_ANCHOR])[0]

    poles = {}
    for fname in available_features:
        desc = feature_desc_map.get(fname, fname)
        feat_emb = embedder.encode([f'{fname}: {desc}'])[0]
        sim_high = 1.0 - cosine_dist(feat_emb, high_anchor_emb)
        sim_low = 1.0 - cosine_dist(feat_emb, low_anchor_emb)
        poles[fname] = {
            'pole': 'HIGH' if sim_high > sim_low else 'LOW',
            'sim_high': round(sim_high, 4),
            'sim_low': round(sim_low, 4),
        }
    return poles


def infer_feature_directions(prompt: str,
                             selected_features: list[tuple[str, float]],
                             feature_poles: dict,
                             embedder) -> dict[str, dict]:
    """Infer per-feature direction by comparing the prompt's pole to each feature's pole.

    - Prompt pole: is the prompt closer to HIGH_ANCHOR or LOW_ANCHOR?
    - If prompt and feature share the same pole → direction = +1 (higher value = better)
    - If opposite poles → direction = -1 (lower value = better)

    Returns {feature_name: {'relevance': float, 'direction': +1 or -1,
                            'prompt_pole': str, 'feature_pole': str, 'confidence': float}}
    """
    prompt_emb = embedder.encode([prompt])[0]
    high_anchor_emb = embedder.encode([HIGH_ANCHOR])[0]
    low_anchor_emb = embedder.encode([LOW_ANCHOR])[0]

    sim_prompt_high = 1.0 - cosine_dist(prompt_emb, high_anchor_emb)
    sim_prompt_low = 1.0 - cosine_dist(prompt_emb, low_anchor_emb)
    prompt_pole = 'HIGH' if sim_prompt_high > sim_prompt_low else 'LOW'
    prompt_confidence = abs(sim_prompt_high - sim_prompt_low)

    results = {}
    for fname, relevance in selected_features:
        if fname not in feature_poles:
            continue
        fpole = feature_poles[fname]['pole']
        same_side = (prompt_pole == fpole)
        direction = 1 if same_side else -1
        feat_confidence = abs(feature_poles[fname]['sim_high'] - feature_poles[fname]['sim_low'])

        results[fname] = {
            'relevance': relevance,
            'direction': direction,
            'prompt_pole': prompt_pole,
            'feature_pole': fpole,
            'confidence': round(min(prompt_confidence, feat_confidence), 4),
            'sim_prompt_high': round(sim_prompt_high, 4),
            'sim_prompt_low': round(sim_prompt_low, 4),
        }

    return results


# Pre-compute feature poles (only needs to be done once)
if SBERT_AVAILABLE:
    feature_poles = compute_feature_poles(FEATURE_CATALOG, AVAILABLE_FEATURES,
                                          FEATURE_DESC_MAP, sbert_model)
    n_high = sum(1 for v in feature_poles.values() if v['pole'] == 'HIGH')
    n_low = len(feature_poles) - n_high
    print(f'Feature poles computed: {n_high} HIGH, {n_low} LOW (total {len(feature_poles)})')

    # Show a sample
    print(f'\nSample feature poles:')
    for fname in ['bpm', 'mood_happy', 'mood_sad', 'mood_relaxed', 'mood_aggressive',
                  'danceability', 'average_loudness', 'onset_rate', 'dynamic_complexity']:
        if fname in feature_poles:
            p = feature_poles[fname]
            print(f'  {fname:<35} {p["pole"]:<5} (h={p["sim_high"]:.4f} l={p["sim_low"]:.4f})')

Feature poles computed: 61 HIGH, 38 LOW (total 99)

Sample feature poles:
  bpm                                 HIGH  (h=0.2934 l=0.2180)
  mood_happy                          HIGH  (h=0.3885 l=0.3146)
  mood_relaxed                        LOW   (h=0.3482 l=0.4321)
  danceability                        HIGH  (h=0.2475 l=0.1891)
  average_loudness                    LOW   (h=0.3181 l=0.3257)
  onset_rate                          HIGH  (h=0.3866 l=0.2998)
  dynamic_complexity                  HIGH  (h=0.4148 l=0.3135)


In [15]:
# Visualize direction inference across all test prompts
if SBERT_AVAILABLE:
    print('Direction inference across all test prompts:\n')
    for prompt in TEST_PROMPTS:
        feats = sbert_feature_select(prompt, feature_embeddings, sbert_model)
        dirs = infer_feature_directions(prompt, feats, feature_poles, sbert_model)

        prompt_pole = list(dirs.values())[0]['prompt_pole'] if dirs else '?'
        print(f'Prompt: "{prompt}"  [pole={prompt_pole}]')
        for fname, info in dirs.items():
            arrow = '↑' if info['direction'] == 1 else '↓'
            print(f'  {arrow} {fname} (feat_pole={info["feature_pole"]}, conf={info["confidence"]:.4f})')
        print()

Direction inference across all test prompts:

Prompt: "energetic tango with strong rhythm for experienced dancers"  [pole=HIGH]
  ↓ style (feat_pole=LOW, conf=0.0074)
  ↑ danceability (feat_pole=HIGH, conf=0.0584)
  ↑ is_danceable (feat_pole=HIGH, conf=0.0487)
  ↑ duration (feat_pole=HIGH, conf=0.0434)
  ↑ jamendo_dance (feat_pole=HIGH, conf=0.1393)
  ↑ bpm (feat_pole=HIGH, conf=0.0754)
  ↑ tempogram_global_mean (feat_pole=HIGH, conf=0.0313)
  ↑ onset_rate (feat_pole=HIGH, conf=0.0868)

Prompt: "melancholic and slow, perfect for a late-night vals"  [pole=LOW]
  ↑ mood_relaxed (feat_pole=LOW, conf=0.0839)
  ↑ style (feat_pole=LOW, conf=0.0074)
  ↑ mirex_mood_rollicking_cheerful_fun_sweet_amiable_good_natured (feat_pole=LOW, conf=0.0678)
  ↑ mirex_mood_literate_poignant_wistful_bittersweet_autumnal_brooding (feat_pole=LOW, conf=0.0852)
  ↑ mirex_mood_humorous_silly_campy_quirky_whimsical_witty_wry (feat_pole=LOW, conf=0.0938)
  ↓ duration (feat_pole=HIGH, conf=0.0434)
  ↓ mood_party (fea

---
## Step 3: Hardcoded Ranking (Percentile + Direction)

For each song, score it on the selected features using the inferred directions:
- Percentile-rank each feature across all songs (0–1)
- If direction = +1 → higher percentile = better score
- If direction = −1 → invert: score = 1 − percentile
- Weight each feature's score by its relevance (cosine similarity from Step 1)
- Sum weighted scores across features, normalize to 0–1

In [16]:
# --- Step 3: Percentile Ranking with Per-Feature Direction ---

def score_songs_sbert_direction(df: pd.DataFrame,
                                feature_directions: dict[str, dict]) -> pd.Series:
    """Score each song using percentile rank + per-feature direction.

    For each feature:
      - direction +1 → percentile as-is (higher value = higher score)
      - direction -1 → 1 - percentile (lower value = higher score)
      - weighted by feature relevance from SBERT similarity

    Returns pd.Series of scores (0-1), indexed like df.
    """
    scores = pd.Series(0.0, index=df.index)
    total_weight = 0.0

    for fname, info in feature_directions.items():
        if fname not in df.columns:
            continue
        col = df[fname]
        if col.dtype not in ['float64', 'float32', 'int64']:
            continue

        # Percentile rank (0-1)
        pct = col.rank(pct=True)

        # Apply direction
        if info['direction'] > 0:
            feature_score = pct
        else:
            feature_score = 1.0 - pct

        weight = info['relevance']
        scores += feature_score * weight
        total_weight += weight

    # Normalize to 0-1
    if total_weight > 0:
        scores /= total_weight
    if scores.max() > scores.min():
        scores = (scores - scores.min()) / (scores.max() - scores.min())

    return scores


print('Scoring function defined.')

Scoring function defined.


---
## Full Pipeline: Method 3 — SBERT Direction

Combine all three steps into a single function that matches the `MatchResult` format used by all other methods.

In [17]:
# --- Full Method 3 Pipeline ---

def rank_songs_sbert_direction(prompt: str, df: pd.DataFrame,
                               feature_embeddings: dict,
                               feature_poles: dict,
                               embedder,
                               top_n_features: int = 8,
                               top_k: int = 10,
                               bottom_k: int = 5) -> MatchResult:
    """Full pipeline for Method 3: SBERT feature selection + anchor-pole direction + percentile ranking.

    Steps:
      1. Select top-N features by SBERT cosine similarity
      2. Infer per-feature direction via anchor-pole comparison
      3. Score songs by weighted percentile rank with inferred directions

    No LLM calls — purely embedding-based.
    """
    t0 = time.time()

    # Step 1: Feature selection
    selected_features = sbert_feature_select(prompt, feature_embeddings, embedder,
                                             top_n=top_n_features)

    # Step 2: Direction inference
    feature_directions = infer_feature_directions(prompt, selected_features,
                                                  feature_poles, embedder)

    # Step 3: Score and rank
    scores = score_songs_sbert_direction(df, feature_directions)
    elapsed = time.time() - t0

    # Build feature_ranges_used in a format compatible with other methods
    feature_ranges_used = {}
    for fname, info in feature_directions.items():
        if fname in df.columns:
            col = df[fname].dropna()
            entry = {
                'direction': 'higher_better' if info['direction'] == 1 else 'lower_better',
                'relevance': round(info['relevance'], 4),
                'prompt_pole': info['prompt_pole'],
                'feature_pole': info['feature_pole'],
                'confidence': info['confidence'],
            }
            # Only add min/max for numeric columns
            if col.dtype in ['float64', 'float32', 'int64']:
                entry['min'] = round(float(col.min()), 4)
                entry['max'] = round(float(col.max()), 4)
            feature_ranges_used[fname] = entry

    # Build top-K and bottom-K song lists
    def _build_songs(indices):
        songs = []
        for idx in indices:
            row = df.loc[idx]
            matched = {}
            for fname in feature_directions:
                if fname in row.index and pd.notna(row[fname]):
                    try:
                        matched[fname] = round(float(row[fname]), 4)
                    except (ValueError, TypeError):
                        pass
            direction_summary = ', '.join(
                f'{fname} {"↑" if info["direction"] == 1 else "↓"}'
                for fname, info in feature_directions.items()
            )
            songs.append({
                'filename': row['filename'],
                'score': round(float(scores[idx]), 4),
                'matched_features': matched,
                'explanation': f'SBERT direction ranking ({direction_summary})',
            })
        return songs

    top_indices = scores.nlargest(top_k).index
    bottom_indices = scores.nsmallest(bottom_k).index

    return {
        'prompt': prompt,
        'method': 'sbert_direction',
        'top_k_songs': _build_songs(top_indices),
        'bottom_k_songs': _build_songs(bottom_indices),
        'feature_ranges_used': feature_ranges_used,
        'metadata': {
            'model': 'all-MiniLM-L6-v2',
            'n_features_selected': len(selected_features),
            'latency_seconds': round(elapsed, 3),
            'uses_llm': False,
        },
    }


print('Method 3 pipeline defined.')

Method 3 pipeline defined.


---
## Run Method 3 Across All Test Prompts

In [18]:
# Run Method 3 across all test prompts
if SBERT_AVAILABLE:
    sbert_dir_results = []
    for prompt in TEST_PROMPTS:
        print(f'\nPrompt: "{prompt}"')
        result = rank_songs_sbert_direction(
            prompt, df_merged, feature_embeddings, feature_poles, sbert_model,
            top_k=TOP_K, bottom_k=BOTTOM_K
        )
        sbert_dir_results.append(result)

        # Print selected features with directions
        print(f'  Features ({result["metadata"]["n_features_selected"]}):')
        for fname, spec in result['feature_ranges_used'].items():
            arrow = '↑' if spec['direction'] == 'higher_better' else '↓'
            print(f'    {arrow} {fname} (relevance={spec["relevance"]:.3f}, '
                  f'pole={spec["feature_pole"]}, conf={spec["confidence"]:.4f})')
        print(f'  Top 3: {[s["filename"] for s in result["top_k_songs"][:3]]}')
        print(f'  Bottom 3: {[s["filename"] for s in result["bottom_k_songs"][:3]]}')
        print(f'  Latency: {result["metadata"]["latency_seconds"]}s')

    ALL_RESULTS['sbert_direction'] = sbert_dir_results
    print(f'\n--- Method 3 complete: {len(sbert_dir_results)} prompts processed ---')
else:
    print('Method 3 skipped — sentence-transformers not available')
    ALL_RESULTS['sbert_direction'] = []


Prompt: "energetic tango with strong rhythm for experienced dancers"
  Features (8):
    ↓ style (relevance=0.660, pole=LOW, conf=0.0074)
    ↑ danceability (relevance=0.525, pole=HIGH, conf=0.0584)
    ↑ is_danceable (relevance=0.481, pole=HIGH, conf=0.0487)
    ↑ duration (relevance=0.445, pole=HIGH, conf=0.0434)
    ↑ jamendo_dance (relevance=0.428, pole=HIGH, conf=0.1393)
    ↑ bpm (relevance=0.402, pole=HIGH, conf=0.0754)
    ↑ tempogram_global_mean (relevance=0.396, pole=HIGH, conf=0.0313)
    ↑ onset_rate (relevance=0.371, pole=HIGH, conf=0.0868)
  Top 3: ['26 Bandera Baja.mp3', '02 Yo Te Bendigo.mp3', '08 Uno.mp3']
  Bottom 3: ['04 La Viruta.mp3', '08 Royal Pigall.mp3', '07 La Morocha.mp3']
  Latency: 0.042s

Prompt: "melancholic and slow, perfect for a late-night vals"
  Features (8):
    ↑ mood_relaxed (relevance=0.325, pole=LOW, conf=0.0839)
    ↑ style (relevance=0.306, pole=LOW, conf=0.0074)
    ↑ mirex_mood_rollicking_cheerful_fun_sweet_amiable_good_natured (relevance=0.

---
## Save Results

In [19]:
# --- Save Method 3 results ---
results_dir = Path(PROCESSED_DIR) / 'features_exp'
results_dir.mkdir(parents=True, exist_ok=True)

if ALL_RESULTS.get('sbert_direction'):
    import copy as _copy

    def _jsonify(obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, dict):
            return {k: _jsonify(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [_jsonify(v) for v in obj]
        return obj

    with open(results_dir / 'sbert_direction_results.json', 'w', encoding='utf-8') as f:
        json.dump(_jsonify(ALL_RESULTS['sbert_direction']), f, indent=2)
    print(f"Saved {len(ALL_RESULTS['sbert_direction'])} Method 3 results to sbert_direction_results.json")
else:
    print('No Method 3 results to save')

Saved 8 Method 3 results to sbert_direction_results.json
